# Step 1: Data Preparation for Ad Generation

**Approach: product-first, fully automated selection**

Products and topics are selected by a deterministic scoring rule — no manual curation.

**Selection rules:**
- *Eligibility:* 2+ topics with 5+ positive AND 5+ negative reviews; non-empty metadata description; exclude non-guitar accessories
- *Product score:* `n_qualifying_topics + description_length / 1000` (richness first, description quality as tiebreaker)
- *Deduplication:* one product per brand (highest score retained)
- *Final selection:* top 3 products
- *Topic selection:* per product, pick 3 qualifying topics with smallest balance gap (|pct_positive − 50%|)

**Output:** `ad_generation_input.json`

## 1. Load Review Data

In [1]:
import pandas as pd
import json

df = pd.read_parquet('../guitars_with_topics_v2.parquet')
print(f'Total reviews: {len(df)}')
df.head(3)

Total reviews: 134068


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,subcategory,store,average_rating,price,topic_id,topic_label,topic_prob
33,3,fun guitar. Very good for beginners,The guitar stayed in tune through many playing...,[],B06ZZG2H35,B06ZZG2H35,AHATA6X6MYTC3VNBFJ3WIYVK257A,2019-05-13 21:42:12.036,0,False,Guitars,Cordoba,4.3,None,7.0,Setup / action,0.234789
41,3,Decent Starter Instrument,Let me start with saying that my teenage daugh...,[],B075P1H3W2,B078HBZBMT,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2018-04-10 10:33:07.834,0,False,Guitars,Sawtooth,3.3,None,4.0,Beginner learning,0.258854
45,4,Great Starter Kit,My fifteen year old daughter has been wanting ...,[],B01G26SEPS,B07711ZF5C,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2016-12-16 23:52:47.000,4,False,Guitars,Gibson,4.2,None,12.0,Accessories,0.324239


## 2. Load Product Metadata

Provides `product_title`, `description`, `features` for LLM prompts.  
Join key: `parent_asin`.

In [ ]:
print('Loading metadata... (may take ~30 seconds)')

meta_lookup = {}
with open('../meta_Musical_Instruments.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        pa = obj.get('parent_asin')
        if pa:
            desc = obj.get('description', '')
            if isinstance(desc, list):
                desc = ' '.join(desc)
            meta_lookup[pa] = {
                'product_title': obj.get('title', ''),
                'description':   desc,
                'features':      obj.get('features', []),
                'store':         obj.get('store', ''),
                'price':         obj.get('price'),
            }

print(f'Metadata entries loaded: {len(meta_lookup)}')

## 3. Derive Sentiment from Rating

- Rating 4–5 → **positive** (Strategy A input)
- Rating 1–2 → **negative** (Strategy B input)
- Rating 3 → dropped

In [3]:
df = df[df['rating'] != 3].copy()
df['sentiment'] = df['rating'].apply(lambda r: 'positive' if r >= 4 else 'negative')

print(f'Reviews after dropping rating=3: {len(df)}')
print(df['sentiment'].value_counts())

Reviews after dropping rating=3: 124243
sentiment
positive    105720
negative     18523
Name: count, dtype: int64


## 4. Find All Qualifying Product × Topic Combinations

A combination qualifies if it has **5+ positive AND 5+ negative** reviews.

In [4]:
MIN_PER_SENTIMENT = 5

combos = []
for (asin, topic), group in df.groupby(['asin', 'topic_label']):
    n_pos = (group['sentiment'] == 'positive').sum()
    n_neg = (group['sentiment'] == 'negative').sum()
    if n_pos >= MIN_PER_SENTIMENT and n_neg >= MIN_PER_SENTIMENT:
        combos.append({
            'asin':        asin,
            'topic':       topic,
            'n_pos':       int(n_pos),
            'n_neg':       int(n_neg),
            'total':       int(n_pos + n_neg),
            'pct_pos':     round(n_pos / (n_pos + n_neg) * 100, 1),
            'balance_gap': round(abs(n_pos / (n_pos + n_neg) - 0.5) * 100, 1),
            'parent_asin': group['parent_asin'].iloc[0],
            'store':       group['store'].iloc[0],
            'avg_rating':  group['average_rating'].iloc[0],
        })

combo_df = pd.DataFrame(combos)
print(f'Qualifying product x topic combinations: {len(combo_df)}')
print(f'Unique products: {combo_df["asin"].nunique()}')
print(f'\nTopics represented:')
print(combo_df['topic'].value_counts())

Qualifying product x topic combinations: 307
Unique products: 132

Topics represented:
topic
Customer service / returns    67
String quality                50
Shipping damage               41
Fret / neck setup             40
Tuning stability              38
Accessories                   28
Beginner learning             26
Playability / chords           6
Electronics / controls         6
Guitar size                    2
Visual appearance              2
Setup / action                 1
Name: count, dtype: int64


## 5. Automated Product Selection

Scoring rule → deduplication → top 3.

In [5]:
# --- Config ---
N_PRODUCTS          = 3    # number of products to select
N_TOPICS_PER_PROD   = 3    # number of topics per product
SAMPLE_SIZE         = 10   # max reviews to sample per topic per sentiment
EXCLUDE_ASINS       = ['B004GEW3H4']  # non-guitar accessories

# Build product-level summary
product_summary = combo_df.groupby('asin').agg(
    n_topics    = ('topic',  'count'),
    store       = ('store',  'first'),
    avg_rating  = ('avg_rating', 'first'),
    parent_asin = ('parent_asin', 'first'),
).reset_index()

product_summary['desc']     = product_summary['parent_asin'].apply(
    lambda pa: meta_lookup.get(pa, {}).get('description', ''))
product_summary['has_desc'] = product_summary['desc'].apply(
    lambda d: bool(d.strip()))
product_summary['desc_len'] = product_summary['desc'].apply(len)
product_summary['title']    = product_summary['parent_asin'].apply(
    lambda pa: meta_lookup.get(pa, {}).get('product_title', ''))

# Apply eligibility filter
eligible = product_summary[
    (product_summary['n_topics'] >= 2) &
    (product_summary['has_desc']) &
    (~product_summary['asin'].isin(EXCLUDE_ASINS))
].copy()

# Score and sort
eligible['score'] = eligible['n_topics'] + eligible['desc_len'] / 1000
eligible = eligible.sort_values('score', ascending=False).reset_index(drop=True)

# One-per-brand deduplication
selected_rows = []
used_brands   = set()
for _, row in eligible.iterrows():
    if row['store'] not in used_brands:
        selected_rows.append(row)
        used_brands.add(row['store'])
    if len(selected_rows) == N_PRODUCTS:
        break

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print(f'=== Selected {N_PRODUCTS} products ===')
print(selected_df[['asin', 'store', 'avg_rating', 'n_topics', 'score',
                    'title']].to_string())

=== Selected 3 products ===
         asin            store  avg_rating  n_topics  score                                                                                                                       title
0  B002RXXOX8  Davison Guitars         4.3         7  8.197  Davison Guitars Full Size Electric Guitar with 10-Watt Amp, Black - Right Handed Beginner Kit with Gig Bag and Accessories
1  B006CYVD5E   Directly Cheap         4.1         6  8.158                                Directly Cheap 6 String Acoustic Guitar Pack, Right Handed, Red (000-BT-GA3810R-RDS+Lessons)
2  B002X49732         Crescent         4.1         6  6.762                          Crescent MG38-CF 38" Acoustic Guitar Starter Package, COFFEE (Includes CrescentTM Digital E-Tuner)


## 6. Automated Topic Selection

For each selected product, pick the `N_TOPICS_PER_PROD` qualifying topics
with the smallest balance gap (pct_positive closest to 50%).

In [6]:
selected_topics = {}  # asin -> list of topic names

print('=== Selected topics per product ===\n')
for _, row in selected_df.iterrows():
    product_combos = combo_df[combo_df['asin'] == row['asin']].sort_values('balance_gap')
    chosen = product_combos.head(N_TOPICS_PER_PROD)
    selected_topics[row['asin']] = chosen['topic'].tolist()

    print(f"{row['asin']} | {row['store']} | {row['title'][:55]}")
    for _, c in chosen.iterrows():
        print(f"  {c['topic']:<35} pct_pos={c['pct_pos']:5.1f}%  "
              f"pos={c['n_pos']:3d}  neg={c['n_neg']:3d}  gap={c['balance_gap']:.1f}pp")
    print()

# Cross-product topic overlap
topic_sets = [set(v) for v in selected_topics.values()]
shared_all = topic_sets[0] & topic_sets[1] & topic_sets[2]
print(f'Topics shared across all 3 products: {shared_all if shared_all else "none"}')

total_ads = sum(len(v) for v in selected_topics.values()) * 2
print(f'\nTotal ads to generate: {total_ads}')

=== Selected topics per product ===

B002RXXOX8 | Davison Guitars | Davison Guitars Full Size Electric Guitar with 10-Watt 
  Tuning stability                    pct_pos= 52.0%  pos= 13  neg= 12  gap=2.0pp
  Fret / neck setup                   pct_pos= 55.2%  pos= 16  neg= 13  gap=5.2pp
  String quality                      pct_pos= 59.3%  pos= 16  neg= 11  gap=9.3pp

B006CYVD5E | Directly Cheap | Directly Cheap 6 String Acoustic Guitar Pack, Right Han
  Customer service / returns          pct_pos= 47.1%  pos= 24  neg= 27  gap=2.9pp
  String quality                      pct_pos= 38.5%  pos= 10  neg= 16  gap=11.5pp
  Tuning stability                    pct_pos= 22.9%  pos= 11  neg= 37  gap=27.1pp

B002X49732 | Crescent | Crescent MG38-CF 38" Acoustic Guitar Starter Package, C
  Customer service / returns          pct_pos= 54.3%  pos= 19  neg= 16  gap=4.3pp
  String quality                      pct_pos= 43.8%  pos= 14  neg= 18  gap=6.2pp
  Fret / neck setup                   pct_pos= 41.

## 7. Sample Reviews and Build Output JSON

For each selected product × topic:
- Up to 10 positive reviews → Strategy A input
- Up to 10 negative reviews → Strategy B input

Sentiment ratios are **product-specific** (not market-wide).

In [7]:
output = []

for _, row in selected_df.iterrows():
    asin      = row['asin']
    pa        = row['parent_asin']
    meta      = meta_lookup.get(pa, {})
    group     = df[df['asin'] == asin]
    topics    = selected_topics[asin]

    product_entry = {
        'asin':           asin,
        'product_title':  meta.get('product_title', ''),
        'brand':          row['store'],
        'average_rating': float(row['avg_rating']) if pd.notna(row['avg_rating']) else None,
        'description':    meta.get('description', ''),
        'features':       meta.get('features', []),
        'topics':         {}
    }

    for topic in topics:
        topic_reviews = group[group['topic_label'] == topic]
        pos_reviews   = topic_reviews[topic_reviews['sentiment'] == 'positive']['text'].dropna()
        neg_reviews   = topic_reviews[topic_reviews['sentiment'] == 'negative']['text'].dropna()

        n_pos = len(pos_reviews)
        n_neg = len(neg_reviews)
        pct_pos = round(n_pos / (n_pos + n_neg) * 100, 1)

        product_entry['topics'][topic] = {
            'pct_positive':     pct_pos,
            'pct_negative':     round(100 - pct_pos, 1),
            'n_positive_total': int(n_pos),
            'n_negative_total': int(n_neg),
            'positive_reviews': pos_reviews.sample(
                min(SAMPLE_SIZE, n_pos), random_state=42).tolist(),
            'negative_reviews': neg_reviews.sample(
                min(SAMPLE_SIZE, n_neg), random_state=42).tolist(),
        }

    output.append(product_entry)
    print(f"Processed: {asin} | {product_entry['product_title'][:60]}")

print(f'\nTotal products: {len(output)}')
print(f'Total ads to generate: {sum(len(p["topics"])*2 for p in output)}')

Processed: B002RXXOX8 | Davison Guitars Full Size Electric Guitar with 10-Watt Amp, 
Processed: B006CYVD5E | Directly Cheap 6 String Acoustic Guitar Pack, Right Handed, 
Processed: B002X49732 | Crescent MG38-CF 38" Acoustic Guitar Starter Package, COFFEE

Total products: 3
Total ads to generate: 18


## 8. Save Output

In [8]:
with open('ad_generation_input.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print('Saved to ad_generation_input.json\n')
print('--- Sanity Check ---')
for p in output:
    print(f"\n{p['asin']} | {p['product_title'][:55]} | ★{p['average_rating']}")
    print(f"  Desc: {p['description'][:120]}...")
    for topic, data in p['topics'].items():
        print(f"  [{topic}] "
              f"{data['n_positive_total']} pos / {data['n_negative_total']} neg total  "
              f"→ sampled {len(data['positive_reviews'])} / {len(data['negative_reviews'])}  "
              f"| pct_pos={data['pct_positive']}%")

Saved to ad_generation_input.json

--- Sanity Check ---

B002RXXOX8 | Davison Guitars Full Size Electric Guitar with 10-Watt  | ★4.3
  Desc: If you're searching for the perfect 6-string electric guitar with a full range of accessories for a student, beginner, i...
  [Tuning stability] 13 pos / 12 neg total  → sampled 10 / 10  | pct_pos=52.0%
  [Fret / neck setup] 16 pos / 13 neg total  → sampled 10 / 10  | pct_pos=55.2%
  [String quality] 16 pos / 11 neg total  → sampled 10 / 10  | pct_pos=59.3%

B006CYVD5E | Directly Cheap 6 String Acoustic Guitar Pack, Right Han | ★4.1
  Desc: This "Guitar" is 38 inches in length and come with Steel Strings. The guitar has Linden Binding and wood construction wi...
  [Customer service / returns] 24 pos / 27 neg total  → sampled 10 / 10  | pct_pos=47.1%
  [String quality] 10 pos / 16 neg total  → sampled 10 / 10  | pct_pos=38.5%
  [Tuning stability] 11 pos / 37 neg total  → sampled 10 / 10  | pct_pos=22.9%

B002X49732 | Crescent MG38-CF 38" Acoustic G

## 9. Preview One Entry

Verify the JSON structure before moving to Step 2 (ad generation).

In [9]:
first       = output[0]
first_topic = list(first['topics'].keys())[0]
data        = first['topics'][first_topic]

print(f"Product : {first['product_title']}")
print(f"Brand   : {first['brand']}  |  Rating: {first['average_rating']}")
print(f"\nDescription:\n{first['description'][:400]}")
print(f"\nFeatures:")
for feat in first['features'][:3]:
    print(f'  • {feat}')

print(f"\n{'='*60}")
print(f"Topic: {first_topic}")
print(f"Product-level sentiment: {data['pct_positive']}% pos / {data['pct_negative']}% neg")
print(f"Total in topic: {data['n_positive_total']} pos, {data['n_negative_total']} neg")

print(f"\n--- Positive reviews (Strategy A input) ---")
for r in data['positive_reviews'][:2]:
    print(f'  • {r[:250]}')

print(f"\n--- Negative reviews (Strategy B input) ---")
for r in data['negative_reviews'][:2]:
    print(f'  • {r[:250]}')

Product : Davison Guitars Full Size Electric Guitar with 10-Watt Amp, Black - Right Handed Beginner Kit with Gig Bag and Accessories
Brand   : Davison Guitars  |  Rating: 4.3

Description:
If you're searching for the perfect 6-string electric guitar with a full range of accessories for a student, beginner, intermediate, or advanced guitar player, the Davison full-size electric guitar is an outstanding choice. Sold as a complete kit with all of the accessories, this guitar package offers excellent quality at a very attractive price. This guitar from Davison is a 39" solid body electr

Features:
  • Complete guitar package: This electric guitar kit includes a top-quality Davison electric guitar with a 10W amp, padded gig bag with backpack straps, shoulder strap, cable, and three picks in assorted colors; everything you need to learn and play guitar!
  • Full size electric guitar: Davison's stunning full-size 39" electric guitar features a sturdy, solid body and exquisite details that mak